In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from scipy.stats import poisson
from sklearn.neighbors import KernelDensity
import time
import dill
from scipy.optimize import minimize
import pickle

/scratch/miniconda/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
metadata = pd.read_csv('../../Allen_institute_labels_v2_classsubclasssupertype.csv')

In [4]:
metadata

,Unnamed: 0,cell_label,cell_barcode,barcoded_cell_sample_label,library_label,feature_matrix_label,entity,brain_section_label,library_method,region_of_interest_acronym,donor_label,donor_genotype,donor_sex,dataset_label,x,y,cluster_alias,subclass_id_label,supertype_id_label,class_id_label
0,0,GCGAGAAGTTAAGGGC-410_B05,GCGAGAAGTTAAGGGC,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.146826,-3.086639,1,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,01 IT-ET Glut
1,1,AATGGCTCAGCTCCTT-411_B06,AATGGCTCAGCTCCTT,411_B06,L8TX_201029_01_E10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550851,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.138481,-3.022000,1,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,01 IT-ET Glut
2,2,AACACACGTTGCTTGA-410_B05,AACACACGTTGCTTGA,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.472557,-2.992709,1,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,01 IT-ET Glut
3,3,CACAGATAGAGGCGGA-410_A05,CACAGATAGAGGCGGA,410_A05,L8TX_201029_01_A10,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.379622,-3.043442,1,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,01 IT-ET Glut
4,4,AAAGTGAAGCATTTCG-410_B05,AAAGTGAAGCATTTCG,410_B05,L8TX_201030_01_C12,WMB-10Xv3-HPF,cell,NaN,10Xv3,RHP,Snap25-IRES2-Cre;Ai14-550850,Ai14(RCL-tdT)/wt,F,WMB-10Xv3,23.909480,-2.601536,1,018 L2 IT PPP-APr Glut,0082 L2 IT PPP-APr Glut_3,01 IT-ET Glut
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4042971,4042971,GTGTGAGCAAACGCGA-1350_C05,GTGTGAGCAAACGCGA,1350_C05,L8XR_220728_01_A05,WMB-10XMulti,cell,NaN,10xRSeq_Mult,MB,C57BL6J-641405,wt/wt,M,WMB-10XMulti,-7.716915,0.223654,8861,278 NLL Gata3 Gly-Gaba,1074 NLL Gata3 Gly-Gaba_1,26 P GABA
4042972,4042972,TTAGCAATCCCTGTTA-1350_C05,TTAGCAATCCCTGTTA,1350_C05,L8XR_220728_01_A05,WMB-10XMulti,cell,NaN,10xRSeq_Mult,MB,C57BL6J-641405,wt/wt,M,WMB-10XMulti,-3.115098,-3.024478,8215,157 RN Spp1 Glut,0682 RN Spp1 Glut_1,19 MB Glut
4042973,4042973,TTTGGCTGTCGCGCAA-1350_C05,TTTGGCTGTCGCGCAA,1350_C05,L8XR_220728_01_A05,WMB-10XMulti,cell,NaN,10xRSeq_Mult,MB,C57BL6J-641405,wt/wt,M,WMB-10XMulti,-7.950964,0.409335,8798,278 NLL Gata3 Gly-Gaba,1076 NLL Gata3 Gly-Gaba_3,26 P GABA
4042974,4042974,ATCCACCTCACAGACT-1320_B04,ATCCACCTCACAGACT,1320_B04,L8XR_220630_02_B10,WMB-10XMulti,cell,NaN,10xRSeq_Mult,OLF,C57BL6J-625156,wt/wt,F,WMB-10XMulti,4.579441,12.135833,8798,278 NLL Gata3 Gly-Gaba,1076 NLL Gata3 Gly-Gaba_3,26 P GABA


In [14]:
hypo = ['15 HY Gnrh1 Glut','16 HY MM Glut','14 HY Glut','13 CNU-HYa Glut','12 HY GABA','11 CNU-HYa GABA']
t0 = time.time()
a = 0
parent_dict = {}
for item in metadata.index:
    a += 1
    parent_dict[metadata.loc[item,'cluster_alias']]= metadata.loc[item,'supertype_id_label']
    parent_dict[metadata.loc[item,'supertype_id_label']] = metadata.loc[item,'subclass_id_label']
    parent_dict[metadata.loc[item,'subclass_id_label']] = metadata.loc[item,'class_id_label']
    if metadata.loc[item,'class_id_label'] in hypo:
        parent_dict[metadata.loc[item,'class_id_label']] = 'hypo'
    else:
        parent_dict[metadata.loc[item,'class_id_label']] = 'not hypo'
    if a % 100000 == 0:
        t1 = time.time()
        print(str(a/len(metadata)) + ' percent in ' + str(t1-t0) + ' seconds')
parent_dict['Unlabeled'] = 'Unlabeled'

0.024734255162533737 percent in 9.753335952758789 seconds
0.04946851032506747 percent in 19.447646141052246 seconds
0.07420276548760121 percent in 29.116336822509766 seconds
0.09893702065013495 percent in 38.826087474823 seconds
0.1236712758126687 percent in 48.55224585533142 seconds
0.14840553097520243 percent in 58.11482548713684 seconds
0.17313978613773617 percent in 67.91018867492676 seconds
0.1978740413002699 percent in 77.6918032169342 seconds
0.22260829646280364 percent in 87.57087421417236 seconds
0.2473425516253374 percent in 97.4849214553833 seconds
0.2720768067878711 percent in 107.1530442237854 seconds
0.29681106195040485 percent in 116.94495391845703 seconds
0.3215453171129386 percent in 126.76430654525757 seconds
0.34627957227547235 percent in 136.59004497528076 seconds
0.37101382743800604 percent in 146.55584931373596 seconds
0.3957480826005398 percent in 156.3852698802948 seconds
0.42048233776307353 percent in 166.30403423309326 seconds
0.4452165929256073 percent in 176

In [15]:
import pickle

with open('../../parent_dict_v2.pkl', 'wb') as f:
    pickle.dump(parent_dict, f)